## 4. Modelagem — descoberta dos regimes

O notebook testa **K-Means de 2 a 8 clusters**. A escolha combina:

- **Silhouette**: maior é melhor;
- **Davies–Bouldin**: menor é melhor;
- **Calinski–Harabasz**: maior é melhor;
- tamanho mínimo do cluster: evitamos tratar um microgrupo como “regime”.

A regra automática escolhe o maior Silhouette entre soluções em que o menor cluster represente pelo menos **3%** das observações.

In [ ]:
cluster_metrics = []
models = {}

for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=30, random_state=RANDOM_STATE)
    labels_k = model.fit_predict(Z_cluster)
    counts = np.bincount(labels_k)

    sil = silhouette_score(
        Z_cluster, labels_k,
        sample_size=min(6000, len(Z_cluster)),
        random_state=RANDOM_STATE
    )
    db = davies_bouldin_score(Z_cluster, labels_k)
    ch = calinski_harabasz_score(Z_cluster, labels_k)
    min_share = counts.min() / len(labels_k)

    cluster_metrics.append((k, sil, db, ch, min_share, counts))
    models[k] = model

html = ['<table><tr><th>k</th><th>Silhouette ↑</th><th>Davies-Bouldin ↓</th><th>Calinski ↑</th><th>Menor regime</th></tr>']
for k, sil, db, ch, min_share, counts in cluster_metrics:
    html.append(
        f'<tr><td>{k}</td><td>{sil:.3f}</td><td>{db:.3f}</td>'
        f'<td>{ch:.0f}</td><td>{100*min_share:.1f}%</td></tr>'
    )
html.append('</table>')
display(HTML(''.join(html)))

valid = [m for m in cluster_metrics if m[4] >= 0.03]
best = max(valid, key=lambda x: x[1]) if valid else max(cluster_metrics, key=lambda x: x[1])
best_k = best[0]
print('Número de macro-regimes selecionado:', best_k)

In [ ]:
ks = [m[0] for m in cluster_metrics]
sils = [m[1] for m in cluster_metrics]
plt.figure(figsize=(7, 4))
plt.plot(ks, sils, marker='o')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Silhouette')
plt.title('Separação dos candidatos a regime')
plt.xticks(ks)
plt.tight_layout()
plt.show()

In [ ]:
regime_model = KMeans(n_clusters=best_k, n_init=50, random_state=RANDOM_STATE)
raw_labels = regime_model.fit_predict(Z_cluster)

intensity_candidates = [
    'CO(GT)', 'PT08.S1(CO)', 'C6H6(GT)', 'PT08.S2(NMHC)',
    'NOx(GT)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)'
]
intensity_idx = [i for i, name in enumerate(feature_names) if name in intensity_candidates]
cluster_intensity = {
    c: float(np.median(Z[raw_labels == c][:, intensity_idx]))
    for c in range(best_k)
}
ordered_raw = sorted(cluster_intensity, key=cluster_intensity.get)
label_map = {raw: new for new, raw in enumerate(ordered_raw, start=1)}
regimes = np.array([label_map[x] for x in raw_labels])

print('Mapeamento de clusters internos -> regimes interpretáveis:', label_map)

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
P = pca.fit_transform(Z_cluster)

plt.figure(figsize=(9, 6))
plt.scatter(P[:, 0], P[:, 1], c=regimes, s=8, alpha=0.45)
plt.xlabel(f'PC1 ({100*pca.explained_variance_ratio_[0]:.1f}% da variância)')
plt.ylabel(f'PC2 ({100*pca.explained_variance_ratio_[1]:.1f}% da variância)')
plt.title('Mapa dos macro-regimes em duas dimensões (PCA)')
plt.tight_layout()
plt.show()